# Módulo 5: Implementación de LoRa en SDR

## Contenido

1. Librerías e implementación base (Módulos 1-4)
2. Receptor óptimo según Vangelista — repaso teórico
3. Loopback digital (referencia, sin hardware)
4. Loopback digital con canal AWGN y canal multipath (Módulos 3 y 4)
5. Conexión y configuración del PlutoSDR
6. Loopback por antena — transmisión y detección real de símbolos
7. Barrido de símbolos y medición de BER con hardware real


## 1. Librerías e implementación base

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- Módulo 1: Codificador / Decodificador ----------
def codificador(bits, SF):
    bits = np.array(bits)
    if len(bits) % SF != 0:
        raise ValueError(f"Cantidad de bits ({len(bits)}) debe ser múltiplo de SF ({SF})")
    num_simbolos = len(bits) // SF
    simbolos = np.zeros(num_simbolos, dtype=int)
    for i in range(num_simbolos):
        bloque = bits[i * SF : (i + 1) * SF]
        valor = 0
        for posicion, bit in enumerate(bloque):
            valor += bit * 2 ** (SF - 1 - posicion)
        simbolos[i] = valor
    return simbolos

def decodificador(simbolos, SF):
    simbolos = np.array(simbolos, dtype=int)
    bits = np.zeros(len(simbolos) * SF, dtype=int)
    for i, simbolo in enumerate(simbolos):
        valor_restante = int(simbolo)
        for posicion in range(SF):
            peso = 2 ** (SF - 1 - posicion)
            if valor_restante >= peso:
                bits[i * SF + posicion] = 1
                valor_restante -= peso
    return bits

def calcular_ber(bits_tx, bits_rx):
    bits_tx = np.array(bits_tx); bits_rx = np.array(bits_rx)
    if len(bits_tx) != len(bits_rx):
        raise ValueError("Los vectores deben tener la misma longitud")
    return np.sum(bits_tx != bits_rx) / len(bits_tx)

def calcular_ser(simbolos_tx, simbolos_rx):
    simbolos_tx = np.array(simbolos_tx); simbolos_rx = np.array(simbolos_rx)
    return np.sum(simbolos_tx != simbolos_rx) / len(simbolos_tx)

# ---------- Módulo 2: Waveform Former / n-Tuple Former ----------
def up_chirp_base(SF, BW, Fs):
    N = 2**SF
    chirp = np.zeros(N, dtype=complex)
    for k in range(N):
        chirp[k] = np.exp(1j * 2 * np.pi * (k**2 / (2*N) - k/2))
    return chirp

def down_chirp(SF, BW, Fs):
    return np.conj(up_chirp_base(SF, BW, Fs))

def waveform_former(simbolos, SF, BW, Fs):
    N = 2**SF
    simbolos = np.array(simbolos)
    cb = up_chirp_base(SF, BW, Fs)
    wf = np.zeros((len(simbolos), N), dtype=complex)
    for i, s in enumerate(simbolos):
        wf[i] = np.roll(cb, -s)
    return wf

def n_tuple_former(waveform, SF, BW, Fs):
    dc = down_chirp(SF, BW, Fs)
    simbolos_rx = np.zeros(waveform.shape[0], dtype=int)
    for i in range(waveform.shape[0]):
        simbolos_rx[i] = np.argmax(np.abs(np.fft.fft(waveform[i] * dc)))
    return simbolos_rx

# ---------- Módulo 3: Ruido AWGN ----------
def agregar_ruido_awgn(señal, snr_db):
    potencia = np.mean(np.abs(señal)**2)
    desviacion = np.sqrt(potencia / (2 * 10**(snr_db/10)))
    ruido = desviacion * (np.random.randn(*señal.shape) + 1j*np.random.randn(*señal.shape))
    return señal + ruido

# ---------- Módulo 4: Canal multipath ----------
def crear_canal_multipath(retardos, atenuaciones, fases_deg):
    longitud = max(retardos) + 1
    h = np.zeros(longitud, dtype=complex)
    for retardo, atenuacion, fase in zip(retardos, atenuaciones, fases_deg):
        h[retardo] = atenuacion * np.exp(1j * fase * np.pi / 180)
    return h

def aplicar_canal_selectivo(waveform, h):
    num_simbolos, N = waveform.shape
    waveform_dist = np.zeros((num_simbolos, N), dtype=complex)
    for i in range(num_simbolos):
        resultado = np.convolve(waveform[i], h, mode='full')
        waveform_dist[i] = resultado[:N]
    return waveform_dist

# ---------- Parámetros del sistema ----------
SF  = 7
BW  = 125e3
Fs  = BW
N   = 2**SF

print("Módulos 1 a 4 cargados correctamente.")
print(f"Parámetros: SF={SF}, BW={BW/1e3:.0f} kHz, N={N} símbolos posibles")

Módulos 1 a 4 cargados correctamente.
Parámetros: SF=7, BW=125 kHz, N=128 símbolos posibles


## 2. El receptor óptimo según Vangelista

El paper de Vangelista (Sección III) deriva el receptor óptimo para FSCM en
un canal AWGN, y demuestra que se puede implementar de forma eficiente en
dos pasos (Ecuación 16):

1. Multiplicar la señal recibida, muestra por muestra, por el down-chirp, obteniendo la señal `d(nTs+kT)`.
2. Calcular la FFT de `d`, y seleccionar el índice `p` que maximiza su módulo.


## 3. Loopback digital

Antes de usar el SDR, confirmamos el comportamiento de referencia: transmitir
y recibir símbolos completamente en software, sin ningún canal.


In [5]:
np.random.seed(42)

num_simbolos_test = 20
bits_tx = np.random.randint(0, 2, num_simbolos_test * SF)
simbolos_tx = codificador(bits_tx, SF)

# Modulación
wf_tx = waveform_former(simbolos_tx, SF, BW, Fs)

# Loopback digital ideal: la señal recibida es idéntica a la transmitida
wf_rx = wf_tx.copy()

# Demodulación (receptor óptimo de Vangelista, Ec. 16)
simbolos_rx = n_tuple_former(wf_rx, SF, BW, Fs)
bits_rx = decodificador(simbolos_rx, SF)

print("=== Loopback digital ideal ===")
print(f"Símbolos TX: {simbolos_tx}")
print(f"Símbolos RX: {simbolos_rx}")
print(f"SER: {calcular_ser(simbolos_tx, simbolos_rx):.4f}")
print(f"BER: {calcular_ber(bits_tx, bits_rx):.4f}")

=== Loopback digital ideal ===
Símbolos TX: [ 34  16  93  63 103  32 125  85  64  27 107  85  23 127  79 122 107  41
  92   0]
Símbolos RX: [ 34  16  93  63 103  32 125  85  64  27 107  85  23 127  79 122 107  41
  92   0]
SER: 0.0000
BER: 0.0000


## 4. Loopback digital con canal AWGN y canal multipath

Como referencia adicional, repetimos el mismo experimento agregando los
canales ya estudiados en los Módulos 3 y 4, para tener un punto de
comparación cuando midamos con hardware real más adelante.


In [7]:
np.random.seed(7)

snr_test = 0   # dB, un caso representativo
bits_tx2 = np.random.randint(0, 2, num_simbolos_test * SF)
simbolos_tx2 = codificador(bits_tx2, SF)
wf_tx2 = waveform_former(simbolos_tx2, SF, BW, Fs)

# --- Solo AWGN ---
wf_awgn = agregar_ruido_awgn(wf_tx2, snr_test)
simbolos_rx_awgn = n_tuple_former(wf_awgn, SF, BW, Fs)
ber_awgn = calcular_ber(bits_tx2, decodificador(simbolos_rx_awgn, SF))

# --- Multipath + AWGN ---
h = crear_canal_multipath(retardos=[0, 5, 10],
                           atenuaciones=[1.0, 0.5, 0.3],
                           fases_deg=[0, 45, 90])
wf_multipath = aplicar_canal_selectivo(wf_tx2, h)
wf_multipath_awgn = agregar_ruido_awgn(wf_multipath, snr_test)
simbolos_rx_mp = n_tuple_former(wf_multipath_awgn, SF, BW, Fs)
ber_mp = calcular_ber(bits_tx2, decodificador(simbolos_rx_mp, SF))

print(f"=== Loopback digital con canal (SNR={snr_test} dB) ===")
print(f"BER solo AWGN        : {ber_awgn:.4f}")
print(f"BER multipath + AWGN : {ber_mp:.4f}")
print()
print("Estos valores sirven como referencia de comparación frente a los")
print("resultados que obtengamos con el hardware SDR real en las secciones")
print("siguientes, donde el canal ya no es simulado sino físico.")

=== Loopback digital con canal (SNR=0 dB) ===
BER solo AWGN        : 0.0000
BER multipath + AWGN : 0.0000

Estos valores sirven como referencia de comparación frente a los
resultados que obtengamos con el hardware SDR real en las secciones
siguientes, donde el canal ya no es simulado sino físico.


## 5. Conexión y configuración del PlutoSDR

> **Prerequisito:** librería `pyadi-iio` instalada (`pip install pyadi-iio`).


In [ ]:
import adi
import time

# ---------- Parámetros RF ----------
PLUTO_IP    = "ip:192.168.2.1"   # Modificar según la IP del laboratorio
FC          = 926e6              # Frecuencia de portadora (banda ISM)
SAMPLE_RATE = 521e3              # Sample rate del Pluto (>= BW)
TX_ATTEN    = -30                # Atenuación TX en dB
RX_GAIN     = 30                 # Ganancia RX en dB

RESAMPLE_FACTOR = int(SAMPLE_RATE / Fs)   # factor de interpolación/decimación

try:
    sdr = adi.Pluto(PLUTO_IP)
    print(f"PlutoSDR conectado en {PLUTO_IP}")
except Exception as e:
    print(f"Error al conectar con el PlutoSDR: {e}")

In [ ]:
def configurar_sdr(sdr, fc, sample_rate, tx_atten, rx_gain):
    """Configura el PlutoSDR para transmisión y recepción de símbolos LoRa."""
    sdr.tx_lo = int(fc)
    sdr.rx_lo = int(fc)
    sdr.sample_rate = int(sample_rate)
    sdr.tx_rf_bandwidth = int(sample_rate)
    sdr.rx_rf_bandwidth = int(sample_rate)
    sdr.tx_hardwaregain_chan0 = tx_atten
    sdr.gain_control_mode_chan0 = 'manual'
    sdr.rx_hardwaregain_chan0 = rx_gain
    sdr.rx_buffer_size = 8 * N * RESAMPLE_FACTOR
    print(f"✅ SDR configurado: Fc={fc/1e6:.0f} MHz, "
          f"Sample rate={sample_rate/1e3:.0f} kHz, "
          f"TX atten={tx_atten} dB, RX gain={rx_gain} dB")

configurar_sdr(sdr, FC, SAMPLE_RATE, TX_ATTEN, RX_GAIN)

## 6. Funciones de resampleo, transmisión y recepción

La señal banda base está a `Fs=BW` pero el Pluto trabaja a `SAMPLE_RATE`,
que es mayor. Hay que interpolar antes de transmitir y decimar después de
recibir para volver a banda base.


In [ ]:
def interpolar(señal, factor):
    """Sube el sample rate repitiendo cada muestra 'factor' veces."""
    return np.repeat(señal, factor)

def decimar(señal, factor):
    """Baja el sample rate tomando 1 de cada 'factor' muestras."""
    return señal[::factor]

def normalizar(señal):
    """Normaliza la amplitud para no saturar el DAC del SDR."""
    max_val = np.max(np.abs(señal))
    return señal / max_val if max_val > 0 else señal

def transmitir_recibir_simbolos(sdr, simbolos, SF, BW, Fs, resample_factor,
                                  num_buffers=4):
    """
    Transmite una secuencia de símbolos LoRa por el PlutoSDR y recibe la
    señal correspondiente (loopback por antena si TX y RX están conectados
    físicamente con un cable).

    Implementa directamente el modulador y el receptor óptimo de Vangelista
    (Ecs. 2 y 16), sin estructura de trama ni preámbulo: se asume que el
    receptor ya sabe dónde empieza la señal (offset fijo tras el buffer
    de transmisión), tal como se plantea en el alcance de este notebook.

    Retorna
    -------
    simbolos_rx : símbolos detectados
    señal_rx    : señal recibida en banda base (para inspección/gráficos)
    """
    N = 2**SF

    # Modulación (Ecuación 2 de Vangelista)
    wf_tx = waveform_former(simbolos, SF, BW, Fs)
    señal_tx = wf_tx.flatten()

    # Interpolación al sample rate del hardware y normalización
    señal_rf = normalizar(interpolar(señal_tx, resample_factor)).astype(np.complex64)

    # Transmisión cíclica
    sdr.tx_cyclic_buffer = True
    sdr.tx(señal_rf)
    time.sleep(0.1)

    # Recepción (se descarta el primer buffer por transitorios de inicio)
    muestras = np.array([], dtype=complex)
    for i in range(num_buffers):
        buf = sdr.rx()
        if i > 0:
            muestras = np.concatenate([muestras, buf])

    sdr.tx_destroy_buffer()

    # Decimación de vuelta a banda base
    señal_rx = decimar(muestras, resample_factor).astype(complex)
    señal_rx = normalizar(señal_rx)

    # Alineación simple: buscamos el punto de máxima energía de correlación
    # con el primer símbolo transmitido, como referencia de inicio
    # (no es un detector de preámbulo -- es solo una búsqueda de offset
    # para poder recortar los N*len(simbolos) muestras del payload recibido)
    dc = down_chirp(SF, BW, Fs)
    mejor_offset, mejor_energia = 0, -1
    max_offset = min(len(señal_rx) - N * len(simbolos), N * 4)
    for offset in range(0, max(max_offset, 1)):
        ventana = señal_rx[offset : offset + N]
        energia = np.max(np.abs(np.fft.fft(ventana * dc)))
        if energia > mejor_energia:
            mejor_energia, mejor_offset = energia, offset

    señal_payload = señal_rx[mejor_offset : mejor_offset + N * len(simbolos)]

    if len(señal_payload) < N * len(simbolos):
        return None, señal_rx

    wf_rx = señal_payload.reshape(len(simbolos), N)
    simbolos_rx = n_tuple_former(wf_rx, SF, BW, Fs)

    return simbolos_rx, señal_rx

print("Funciones de transmisión/recepción definidas.")

## 7. Loopback por antena — transmisión y detección real de símbolos

In [ ]:
print("=== Loopback por antena ===")
print("Verificá que el cable SMA conecte TX con RX antes de continuar.\n")

try:
    np.random.seed(123)
    num_simbolos_hw = 15
    bits_hw = np.random.randint(0, 2, num_simbolos_hw * SF)
    simbolos_hw = codificador(bits_hw, SF)

    simbolos_rx_hw, señal_rx_hw = transmitir_recibir_simbolos(
        sdr, simbolos_hw, SF, BW, Fs, RESAMPLE_FACTOR)

    if simbolos_rx_hw is None:
        print("❌ No se pudo recuperar el payload completo. "
              "Verificá la conexión, TX_ATTEN o RX_GAIN.")
    else:
        bits_rx_hw = decodificador(simbolos_rx_hw, SF)
        ser_hw = calcular_ser(simbolos_hw, simbolos_rx_hw)
        ber_hw = calcular_ber(bits_hw, bits_rx_hw)

        print(f"Símbolos TX : {simbolos_hw}")
        print(f"Símbolos RX : {simbolos_rx_hw}")
        print(f"SER         : {ser_hw:.4f}")
        print(f"BER         : {ber_hw:.4f}")

except Exception as e:
    print(f"❌ Error durante la transmisión/recepción: {e}")

## 8. Visualización de la señal recibida por hardware

Comparamos el espectrograma de la señal efectivamente recibida por el SDR
contra lo esperado teóricamente, para verificar visualmente la forma de
onda chirp tras pasar por el hardware real.


In [ ]:
try:
    if 'señal_rx_hw' in dir() and señal_rx_hw is not None:
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.specgram(señal_rx_hw, NFFT=N//4, Fs=Fs, noverlap=N//8,
                    cmap='inferno', sides='twosided')
        ax.set_title("Espectrograma de la señal recibida por el PlutoSDR (loopback por antena)")
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Frecuencia (Hz)")
        ax.set_ylim(-BW/2*1.1, BW/2*1.1)
        plt.tight_layout()
        plt.show()
    else:
        print("Corré primero la sección 7 (loopback por antena).")
except Exception as e:
    print(f"Error al graficar: {e}")

## 9. Barrido de símbolos y medición de BER con hardware real

Repetimos la transmisión/recepción variando la atenuación TX, para
caracterizar cómo se comporta el receptor óptimo de Vangelista frente a
distintos niveles de potencia recibida, usando el hardware real como canal.


In [ ]:
atenuaciones_tx = [-10, -20, -30, -40, -50]
ber_por_atenuacion = []

print(f"{'TX Atten (dB)':>15} | {'BER':>10} | {'Estado':>10}")
print("-" * 42)

try:
    for atten in atenuaciones_tx:
        sdr.tx_hardwaregain_chan0 = atten

        np.random.seed(456)
        bits_b = np.random.randint(0, 2, 20 * SF)
        simbolos_b = codificador(bits_b, SF)

        simbolos_rx_b, _ = transmitir_recibir_simbolos(
            sdr, simbolos_b, SF, BW, Fs, RESAMPLE_FACTOR)

        if simbolos_rx_b is None:
            ber_por_atenuacion.append(0.5)
            print(f"{atten:>15} | {'—':>10} | {'❌':>10}")
            continue

        bits_rx_b = decodificador(simbolos_rx_b, SF)
        ber_b = calcular_ber(bits_b, bits_rx_b)
        ber_por_atenuacion.append(ber_b)
        print(f"{atten:>15} | {ber_b:>10.4f} | {'✅':>10}")

    ber_por_atenuacion = np.array(ber_por_atenuacion)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(atenuaciones_tx,
                np.where(ber_por_atenuacion > 0, ber_por_atenuacion, 1e-5),
                marker='o', markersize=6, linewidth=1.5, color='steelblue')
    ax.set_xlabel("Atenuación TX (dB)")
    ax.set_ylabel("BER")
    ax.set_title("BER vs Atenuación TX — Loopback por antena (hardware real)")
    ax.grid(True, which='both', alpha=0.4)
    ax.invert_xaxis()
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"❌ Error durante el barrido: {e}")
    print("   Verificá la conexión del loopback por antena.")